In [7]:
import pandas as pd

df_train = pd.read_csv('../../data/raw/train.csv')
df_test = pd.read_csv('../../data/raw/test.csv')
df_submission = pd.read_csv('../../data/raw/sample_submission.csv')

In [8]:
# 1. simple feature engineering

df_train_original = df_train.loc[df_train['type'] == 'original']
df_train_original.drop(columns=['type'], inplace=True)

df_train_original.loc[df_train_original['review'].isnull(), 'review'] = ' '
df_test.loc[df_test['review'].isnull(), 'review'] = ' '

C:\Users\pilla\AppData\Local\Temp\ipykernel_3768\3222876139.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_original.drop(columns=['type'], inplace=True)


In [9]:
# 2. training

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "klue/roberta-base"

epoch = 1
batch_size = 32
batch_size_inference = 128

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4, device_map='auto')
optimizer = torch.optim.AdamW(model.parameters(), lr=2.0e-5, weight_decay=0.001)
scaler = torch.amp.GradScaler()

total_steps = epoch * (len(df_train_original) // batch_size)
scheduler_warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.05, end_factor=1.0, total_iters=total_steps//5)
scheduler_decay = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.25, total_iters=total_steps-(total_steps//5))
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_decay], milestones=[total_steps//5])


model.train()
for t in range(epoch):
    print(f"------------------------ Epoch {t+1} ------------------------")
    total = 0
    correct = 0
    for i in range(0, (len(df_train_original) // batch_size) * batch_size, batch_size):
        batch = df_train_original.iloc[i:i+batch_size]
        texts = batch['review'].tolist()
        labels = batch['label'].tolist()

        encoded = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
        input_ids = encoded['input_ids'].to(model.device)
        attention_mask = encoded['attention_mask'].to(model.device)
        labels = torch.tensor(labels).to(model.device)

        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.75)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        preds = outputs.logits.argmax(dim=1)
        total += len(labels)
        correct += (preds == labels).sum()
        if (i+batch_size) % (batch_size*200) == 0:
            accuracy = correct / total
            print(f"Step {i//batch_size+1}, Loss: {loss.item()}, Train Accuracy: {accuracy.item():.4f}")
            total = 0
            correct = 0

ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

In [ ]:
# 3. preparing to submit

model.eval()
for i in range(0, len(df_test), batch_size_inference):
    batch = df_test.iloc[i:i+batch_size_inference]
    texts = batch['review'].tolist()

    encoded = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
    input_ids = encoded['input_ids'].to(model.device)
    attention_mask = encoded['attention_mask'].to(model.device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

    df_submission.loc[batch.index, 'pred'] = logits.argmax(dim=1).cpu().numpy()

df_submission['pred'] = df_submission['pred'].astype(int)
df_submission.to_csv('../../output/submission_5minutes_training_recipe.csv', index=False)